### SUMMARY
- Number of simulations processed: 256
- Min simulated time: 1.04 ms
- Max simulated time: 2.11 ms

### ESTIMATED COMPUTATIONAL RESOURCES:
- Number of restarts: 21
- Total node-hours: 16.9 knode-hours
- Total GPUhs: 67.5 kGPU-hours
- Total wall clock time: 5.4 Days

### DATASET
- ~80 TB of data generated

(run `check_computational_cost.py` to reproduce)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import h5py

from src.scanmetadata import ScanMetadata
tmp_figdir = '/Users/ahoffman/personal_gkyl_scripts/notebooks/figures/'
manuscript_figdir = '/Users/ahoffman/writing/gkyl_tcv_miller_scan/figures/'
h5file = 'data/tcv_miller_scan_big_metadata_frame_500_navg_25.h5'
miller_scan = ScanMetadata(h5file)
miller_scan.info()

In [ ]:
with h5py.File(h5file, 'r') as f:
    
    fig, axs = plt.subplots(2,2)
    axs = axs.flatten()
    scan_table = [key for key in f.keys()]
    colors = ['C0', 'C1', 'C2', 'C3']
    for n in range(0, len(scan_table), 4):
        ic = 0
        for k in range(4):
            ax = axs[k]
            for key in scan_table[n+k:n+k+1]:
                y = 0
                y += f[key]['intmom']['bflux_x_u_He']['values'][:]
                y += f[key]['intmom']['bflux_z_u_He']['values'][:]
                y += f[key]['intmom']['bflux_z_l_He']['values'][:]
                y += f[key]['intmom']['bflux_x_u_Hi']['values'][:]
                y += f[key]['intmom']['bflux_z_u_Hi']['values'][:]
                y += f[key]['intmom']['bflux_z_l_Hi']['values'][:]   
                
                y = y/y[0]
                
                t = f[key]['intmom']['bflux_x_u_He']['time'][:]/1000
            
                ax.plot(t, y, label=key, alpha=0.1, color=colors[ic])
                ic += 1
            ax.set_xlabel('ms')
            ax.set_ylabel('MW')
            ax.set_ylim([0,2.0])
fig.tight_layout()

In [ ]:
figdir = tmp_figdir

with h5py.File(h5file, 'r') as f:
    
    fig, axs = plt.subplots(2,2)
    axs = axs.flatten()
    scan_table = [key for key in f.keys()]
    colors = ['C0', 'C1', 'C2', 'C3']
    for n in range(0, len(scan_table), 4):
        ic = 0
        for k in range(4):
            for key in scan_table[n+k:n+k+1]:
                y = 0
                y += f[key]['intmom']['We']['values'][:] 
                y += f[key]['intmom']['Wi']['values'][:] 
                t = f[key]['intmom']['We']['time'][:]/1000
                
                y = np.gradient(y, t)*1000
                
                # smooth y with a moving average
                window_size = 25
                y = np.convolve(y, np.ones(window_size)/window_size, mode='valid')
                t = t[window_size-1:]
            
                axs[k].plot(t, y, label=key, alpha=0.1, color=colors[ic])
                ic += 1
            # ax.set_xlim([0,1])
            axs[k].set_ylim([-1.0,1.0])

for ax in axs:
    ax.set_xlabel(r'$t$ [ms]')
    ax.set_ylabel(r'$dW/dt$ [MW]')
fig.tight_layout()
fig.savefig(figdir+'energy_time_traces.png', dpi=300)

In [ ]:
with h5py.File(h5file, 'r') as f:
    
    fig, axs = plt.subplots(2,2)
    axs = axs.flatten()
    scan_table = [key for key in f.keys()]
    colors = ['C0', 'C1', 'C2', 'C3']
    for n in range(0, len(scan_table), 4):
        ic = 0
        for k in range(4):
            ax = axs[k]
            for key in scan_table[n+k:n+k+1]:
                y = 0
                y += f[key]['intmom']['ne']['values'][:] 
                y = y/y[0]
                t = f[key]['intmom']['ne']['time'][:]/1000
            
                ax.plot(t, y, label=key, alpha=0.1, color=colors[ic])
                ic += 1
fig.tight_layout()

In [ ]:
figdir = tmp_figdir

with h5py.File(h5file, 'r') as f:
    
    fig, ax = plt.subplots(1,1, figsize=(5,3.5))
    scan_table = [key for key in f.keys()]
    colors = ['C0', 'C1', 'C2', 'C3']
    for n in range(0, len(scan_table), 4):
        ic = 0
        for k in range(4):
            for key in scan_table[n+k:n+k+1]:
                y = 0
                y += f[key]['intmom']['We']['values'][:] 
                y += f[key]['intmom']['Wi']['values'][:] 
                t = f[key]['intmom']['We']['time'][:]/1000
                
                y = y/y[0]
            
                ax.plot(t, y, label=key, alpha=0.25, color=colors[ic])
                ic += 1
            # ax.set_xlim([0,1])
            ax.set_ylim([0,2.0])

ax.set_xlabel(r'$t$ [ms]')
ax.set_ylabel(r'$W/W_0$')
fig.tight_layout()
fig.savefig(figdir+'energy_time_traces.png', dpi=300)

In [ ]:
miller_scan.plot_field_vs_field('energy_srcCORE', 'bflux_wall_H', alpha=0.5)


In [ ]:
lines = []
lines.append({'x': [10,100], 'y': [30,300], 'kwargs': {'color': 'k', 'linestyle':'dashed'}})

miller_scan.plot_field_vs_field('Te_limup', 'phi_limup', alpha=0.5, lines=lines)
miller_scan.plot_field_vs_field('Te_limlo', 'phi_limlo', alpha=0.5, lines=lines)

In [ ]:
lines = []
lines.append({'x': [0,1], 'y': [0,1], 'kwargs': {'color': 'k', 'linestyle':'dashed'}})
lines.append({'x': [0,1], 'y': [0,1.5], 'kwargs': {'color': 'k', 'linestyle':'dashed'}})
miller_scan.plot_field_vs_field('bflux_z_l_H', 'bflux_z_u_H', alpha=0.5, xlim=[0.1,0.55], ylim=[0.1,0.55], lines=lines)

In [ ]:
lines = []
lines.append({'x': [0,1], 'y': [0.1,3.3], 'kwargs': {'color': 'k', 'linestyle':'dashed'}})
miller_scan.plot_field_vs_field('bflux_x_u_H', 'bflux_z_u_H', alpha=0.5, lines=lines)

In [ ]:
miller_scan.plot_field_vs_field('bflux_z_u_H', 'lambda_q', alpha=0.5, lines=[])


In [ ]:
miller_scan.plot_field_vs_field('Pi_norm_limup', 'Pi_norm_edge', alpha=0.5, lines=[])
miller_scan.plot_field_vs_field('ne_norm_limup', 'ne_norm_edge', alpha=0.5, lines=[])
miller_scan.plot_field_vs_field('Ti_norm_limup', 'Ti_norm_edge', alpha=0.5, lines=[])

In [ ]:
figdir = tmp_figdir
miller_scan.plot_field_vs_field('Ti_norm_limup', 'Ti_norm_edge', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, 
                                figfilename=figdir+'Ti_limup_vs_Ti_edge.png',dpi=300)

In [ ]:
figdir = tmp_figdir

miller_scan.plot_field_vs_field('Ti_norm_sol', 'Ti_norm_edge', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, figfilename=figdir+'Ti_sol_vs_Ti_edge.png',dpi=300)
miller_scan.plot_field_vs_field('ne_norm_sol', 'ne_norm_edge', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, figfilename=figdir+'ne_sol_vs_ne_edge.png',dpi=300)
miller_scan.plot_field_vs_field('Pi_norm_sol', 'Pi_norm_edge', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, figfilename=figdir+'pi_sol_vs_pi_edge.png',dpi=300)

miller_scan.plot_field_vs_field('kne', 'kTi', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, figfilename=figdir+'kn_vs_kTi_edge.png',dpi=300)
miller_scan.plot_field_vs_field('kTi', 'kTe', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, figfilename=figdir+'kTi_vs_kTe_edge.png',dpi=300)
miller_scan.plot_field_vs_field('tau_sol', 'tau_edge', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, figfilename=figdir+'tau_sol_vs_tau_edge.png',dpi=300)

In [ ]:
figdir = tmp_figdir

lines = []
shadows = []
ymin = -1/10 - 1/3
ymax = -1/10 + 1.3
xmin = 1/4
xmax = 1
shadows = [{'x': [xmin,xmax], 'y': [ymin,ymax], 
            'kwargs': {'color': 'y', 'alpha': 0.1, 'linestyle':'dashed'}}]
miller_scan.plot_field_vs_field('chixe_over_chixi', 'De_over_chitot', powers=[1e5,5e5, 1e6, 5e6], alpha=0.5, 
                                figfilename=figdir+'instability_classification.png',dpi=300,
                                xlim = [-0., 0.85], ylim=[-0.29,0], lines=lines, shadows=shadows)

In [ ]:
figdir = tmp_figdir

power = 0.1
fixed_param = {'energy_srcCORE': power*1e6}
method = 'contourf'  # Use pcolormesh for better handling of log-scale axes
miller_scan.plot_contour_grid(['avg_dt'], fixed_params=fixed_param, method=method, figfilename=f'{figdir}avg_dt_contour_grid_power_{power}MW.png')

In [ ]:
power = 5e6
miller_scan.plot_contour_grid(['chixe_over_chixi'], fixed_params={'energy_srcCORE': power}, method='pcolormesh',
                              cmap='coolwarm', clim=[0.1,10])
miller_scan.plot_contour_grid(['De_over_chitot'], fixed_params={'energy_srcCORE': power}, method='pcolormesh',
                              cmap='coolwarm', clim=[-1/10-1/3,1/10+1/3])

In [ ]:
# Draw a poloidal cut of a flux surface given kappa and delta
kappa = 1.0
delta = 0.6

theta = np.linspace(-np.pi, np.pi, 100)
R = lambda r: 1 + r*np.cos(theta + np.arcsin(delta)*np.sin(theta))
Z = lambda r: r*kappa*np.sin(theta)

# Create a banana orbit trajectory
r_ban = 0.2  # the banana orbit width
theta_ban = np.pi/3

# Poloidal angle range spanning both tips
theta_orbit = np.linspace(-theta_ban, theta_ban, 200)

# Orbit half-width: zero at the tips (theta = ±theta_ban), maximum r_ban at the outboard (theta = 0)
# Derived from the trapping condition: delta_r ~ sqrt(cos(theta) - cos(theta_ban))
delta_r = r_ban * np.sqrt(
    np.maximum(0, (np.cos(theta_orbit) - np.cos(theta_ban)) / (1 - np.cos(theta_ban)))
)

# Outer leg: particle drifting at r = 1 + delta_r (outward on outboard side)
R_ban_outer = 1 + (1 + delta_r) * np.cos(theta_orbit + np.arcsin(delta) * np.sin(theta_orbit))
Z_ban_outer = kappa * (1 + delta_r) * np.sin(theta_orbit)

# Inner leg: return trajectory at r = 1 - delta_r (traversed in reverse to close the orbit)
R_ban_inner = 1 + (1 - delta_r[::-1]) * np.cos(theta_orbit[::-1] + np.arcsin(delta) * np.sin(theta_orbit[::-1]))
Z_ban_inner = kappa * (1 - delta_r[::-1]) * np.sin(theta_orbit[::-1])

R_ban = np.concatenate([R_ban_outer, R_ban_inner])
Z_ban = np.concatenate([Z_ban_outer, Z_ban_inner])



fig, ax = plt.subplots(figsize=(5,3.5))
ax.plot(R(1), Z(1), label='flux surface', color='C0')
ax.plot(R(0.9), Z(0.9), label='flux surface', color='C0')
ax.plot(R(0.8), Z(0.8), label='flux surface', color='C0')
ax.plot(R_ban, Z_ban, label='banana orbit', color='r')
## Create a shaded area at the high field side of the flux surface
ax.add_patch(plt.Rectangle((-0.1, -0.5), 0.3, 1, color='b', alpha=0.1, label='high-field side'))
## Add the limiter
ax.add_patch(plt.Rectangle((-0.1, -0.05), 0.2, 0.1, color='k', alpha=1.0, label='limiter'))
ax.axis('equal')
# remove the axis
# ax.set_xticks([])
# ax.set_yticks([])
# remove the box
ax.set_frame_on(False)
fig.tight_layout()

In [ ]:
# Draw a poloidal cut of a flux surface given kappa and delta
kappa = 1.0
delta = -0.6

theta = np.linspace(-np.pi, np.pi, 100)
R = lambda r: 1 + r*np.cos(theta + np.arcsin(delta)*np.sin(theta))
Z = lambda r: r*kappa*np.sin(theta)

# Create a banana orbit trajectory
r_ban = 0.2  # the banana orbit width
theta_ban = np.pi*0.68  # the angle of the tip of the banana orbit (inboard midplane)

# Poloidal angle range spanning both tips
theta_orbit = np.linspace(-theta_ban, theta_ban, 200)

# Orbit half-width: zero at the tips (theta = ±theta_ban), maximum r_ban at the outboard (theta = 0)
# Derived from the trapping condition: delta_r ~ sqrt(cos(theta) - cos(theta_ban))
delta_r = r_ban * np.sqrt(
    np.maximum(0, (np.cos(theta_orbit) - np.cos(theta_ban)) / (1 - np.cos(theta_ban)))
)

# Outer leg: particle drifting at r = 1 + delta_r (outward on outboard side)
R_ban_outer = 1 + (1 + delta_r) * np.cos(theta_orbit + np.arcsin(delta) * np.sin(theta_orbit))
Z_ban_outer = kappa * (1 + delta_r) * np.sin(theta_orbit)

# Inner leg: return trajectory at r = 1 - delta_r (traversed in reverse to close the orbit)
R_ban_inner = 1 + (1 - delta_r[::-1]) * np.cos(theta_orbit[::-1] + np.arcsin(delta) * np.sin(theta_orbit[::-1]))
Z_ban_inner = kappa * (1 - delta_r[::-1]) * np.sin(theta_orbit[::-1])

R_ban = np.concatenate([R_ban_outer, R_ban_inner])
Z_ban = np.concatenate([Z_ban_outer, Z_ban_inner])



fig, ax = plt.subplots(figsize=(5,3.5))
ax.plot(R(1), Z(1), label='flux surface', color='C0')
ax.plot(R(0.9), Z(0.9), label='flux surface', color='C0')
ax.plot(R(0.8), Z(0.8), label='flux surface', color='C0')
ax.plot(R_ban, Z_ban, label='banana orbit', color='r')
## Create a shaded area at the high field side of the flux surface
ax.add_patch(plt.Rectangle((-0.1, -0.5), 0.6, 1, color='b', alpha=0.1, label='high-field side'))
## Add the limiter
ax.add_patch(plt.Rectangle((-0.1, -0.05), 0.5, 0.1, color='k', alpha=1.0, label='limiter'))
ax.axis('equal')
# remove the axis
# ax.set_xticks([])
# ax.set_yticks([])
# remove the box
ax.set_frame_on(False)
fig.tight_layout()